# 🔥 TMAX vs TMEAN Heat Risk Comparison

This notebook implements **both TMAX and TMEAN-based heat risk calculations** for comparison:
1. **TMAX approach** - Traditional maximum temperature heat days
2. **TMEAN approach** - Mean temperature heat days for different risk profile
3. **Direct comparison** - Side-by-side maps and analysis

## Key Research Question:
How do heat risk patterns differ when using:
- ⚡ **TMAX** - Captures peak daily heat exposure
- 🌡️ **TMEAN** - Captures sustained heat throughout the day
- 🔍 **Comparison** - Which approach better identifies vulnerable areas?

In [1]:
# Import required libraries
import ee
import geemap
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import rasterio
from tkinter import filedialog
import tkinter as tk

# Set matplotlib to display inline
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

# Initialize Earth Engine with your project
try:
    ee.Initialize(project='tl-cities')
    print('✅ Earth Engine initialized successfully')
except Exception as e:
    print(f'❌ Earth Engine initialization failed: {e}')

print('📦 Available packages:')
print(f'   - xarray: {xr.__version__}')
print(f'   - pandas: {pd.__version__}')
print(f'   - numpy: {np.__version__}')

# Create outputs directory
os.makedirs('../outputs', exist_ok=True)
print('📁 Created outputs directory: ../outputs')

✅ Earth Engine initialized successfully
📦 Available packages:
   - xarray: 2024.7.0
   - pandas: 2.3.1
   - numpy: 1.26.4
📁 Created outputs directory: ../outputs


## 🎯 Step 1: ROI Selection (Same as Notebook 18)

In [2]:
# Global variables
analysis_geom = None
tmax_data = None
tmean_data = None
heat_risk_results = {}

# Create map for ROI selection (same as notebooks 16/17/18)
m = geemap.Map(center=[-12.9714, -38.5014], zoom=10)  # Salvador, Brazil
m.add_basemap('SATELLITE')
m.add('draw_control')

def set_roi_from_drawing():
    '''Extract ROI from map drawing'''
    global analysis_geom
    
    try:
        if hasattr(m, 'draw_control') and len(m.draw_control.data) > 0:
            feature = m.draw_control.data[-1]
            coords = feature['geometry']['coordinates']
            
            if feature['geometry']['type'] == 'Polygon':
                analysis_geom = ee.Geometry.Polygon(coords)
            elif feature['geometry']['type'] == 'Rectangle':
                analysis_geom = ee.Geometry.Rectangle(coords)
            
            area_km2 = analysis_geom.area().divide(1000000).getInfo()
            bounds_info = analysis_geom.bounds().getInfo()['coordinates'][0]
            west, south = bounds_info[0]
            east, north = bounds_info[2]
            
            print(f'✅ ROI set from drawing: {area_km2:.1f} km²')
            print(f'   Bounds: W={west:.3f}, E={east:.3f}, S={south:.3f}, N={north:.3f}')
            return True
        else:
            print('❌ No drawing found. Please draw a polygon or rectangle on the map.')
            return False
    except Exception as e:
        print(f'❌ Error setting ROI from drawing: {e}')
        return False

def set_roi_from_coordinates():
    '''Set ROI from coordinate inputs'''
    global analysis_geom
    
    try:
        west = float(west_input.value) if west_input.value else -38.7
        east = float(east_input.value) if east_input.value else -38.3
        south = float(south_input.value) if south_input.value else -13.1
        north = float(north_input.value) if north_input.value else -12.8
        
        analysis_geom = ee.Geometry.Rectangle([west, south, east, north])
        area_km2 = analysis_geom.area().divide(1000000).getInfo()
        
        roi_image = ee.Image().paint(analysis_geom, 1, 2)
        m.addLayer(roi_image, {'palette': ['red'], 'max': 1}, 'ROI')
        m.centerObject(analysis_geom, 11)
        
        print(f'✅ ROI set from coordinates: {area_km2:.1f} km²')
        print(f'   Bounds: W={west:.3f}, E={east:.3f}, S={south:.3f}, N={north:.3f}')
        return True
    except Exception as e:
        print(f'❌ Error setting ROI from coordinates: {e}')
        return False

def browse_raster_file():
    '''Open file browser to select raster file'''
    try:
        root = tk.Tk()
        root.withdraw()
        
        file_path = filedialog.askopenfilename(
            title='Select Reference Raster File',
            filetypes=[
                ('Raster files', '*.tif *.tiff *.img *.nc *.hdf *.jp2'),
                ('GeoTIFF', '*.tif *.tiff'),
                ('NetCDF', '*.nc'),
                ('All files', '*.*')
            ]
        )
        
        root.destroy()
        
        if file_path:
            raster_path_display.value = file_path
            print(f'📁 Selected file: {os.path.basename(file_path)}')
            return file_path
        else:
            print('❌ No file selected')
            return None
            
    except Exception as e:
        print(f'❌ Error opening file browser: {e}')
        return None

def set_roi_from_raster():
    '''Set ROI from selected raster extent with proper CRS handling'''
    global analysis_geom
    
    try:
        raster_path = raster_path_display.value.strip()
        
        if not raster_path or not os.path.exists(raster_path):
            print('❌ Please select a valid raster file first')
            return False
        
        print(f'📖 Reading raster: {os.path.basename(raster_path)}')
        
        with rasterio.open(raster_path) as src:
            bounds = src.bounds
            crs = src.crs
            
            west, south, east, north = bounds.left, bounds.bottom, bounds.right, bounds.top
            
            print(f'   📊 Original CRS: {crs}')
            print(f'   📊 Original bounds: W={west:.3f}, E={east:.3f}, S={south:.3f}, N={north:.3f}')
            
            # Transform to WGS84 if needed
            if crs.to_epsg() != 4326:
                from rasterio.warp import transform_bounds
                west, south, east, north = transform_bounds(
                    crs, 'EPSG:4326', west, south, east, north
                )
                print(f'   🔄 Transformed to WGS84: W={west:.6f}, E={east:.6f}, S={south:.6f}, N={north:.6f}')
        
        # Create geometry
        analysis_geom = ee.Geometry.Rectangle([west, south, east, north], 'EPSG:4326')
        area_km2 = analysis_geom.area().divide(1000000).getInfo()
        
        roi_image = ee.Image().paint(analysis_geom, 1, 2)
        m.addLayer(roi_image, {'palette': ['blue'], 'max': 1}, 'Raster ROI')
        m.centerObject(analysis_geom, 11)
        
        print(f'   ✅ ROI set from raster extent: {area_km2:.1f} km²')
        return True
        
    except Exception as e:
        print(f'❌ Error setting ROI from raster: {e}')
        return False

# ROI input widgets
west_input = widgets.FloatText(value=-38.7, description='West:')
east_input = widgets.FloatText(value=-38.3, description='East:')
south_input = widgets.FloatText(value=-13.1, description='South:')
north_input = widgets.FloatText(value=-12.8, description='North:')

raster_path_display = widgets.Text(
    value='',
    placeholder='No file selected...',
    description='Selected File:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px'),
    disabled=True
)

browse_button = widgets.Button(description='📂 Browse Files', button_style='info')
browse_button.on_click(lambda b: browse_raster_file())

# Action buttons
set_drawing_button = widgets.Button(description='📍 Use Drawing', button_style='success')
set_coords_button = widgets.Button(description='📍 Use Coordinates', button_style='info')
set_raster_button = widgets.Button(description='📍 Use Raster Extent', button_style='warning')

set_drawing_button.on_click(lambda b: set_roi_from_drawing())
set_coords_button.on_click(lambda b: set_roi_from_coordinates())
set_raster_button.on_click(lambda b: set_roi_from_raster())

roi_interface = widgets.VBox([
    widgets.HTML('<h3>🎯 ROI Selection</h3>'),
    widgets.HTML('<b>Method 1: Draw on Map</b>'),
    set_drawing_button,
    widgets.HTML('<b>Method 2: Enter Coordinates</b>'),
    widgets.HBox([west_input, east_input]),
    widgets.HBox([south_input, north_input]),
    set_coords_button,
    widgets.HTML('<b>Method 3: Use Raster File Extent</b>'),
    widgets.HBox([browse_button, raster_path_display]),
    set_raster_button
])

display(roi_interface)
display(m)

print('🎯 ROI Selection Ready')

Map(center=[-12.9714, -38.5014], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

🎯 ROI Selection Ready


## 📊 Step 2: Analysis Configuration

In [5]:
# Analysis configuration
analysis_year = widgets.IntSlider(value=2020, min=2003, max=2020, description='Analysis Year:')
reference_start = widgets.IntSlider(value=2010, min=2003, max=2019, description='Reference Start:')
reference_end = widgets.IntSlider(value=2019, min=2004, max=2020, description='Reference End:')
absolute_threshold = widgets.FloatSlider(value=35.0, min=20.0, max=45.0, step=0.5, description='Threshold (°C):')
percentile_threshold = widgets.FloatSlider(value=95.0, min=50.0, max=99.0, step=1.0, description='Percentile:')

# TMEAN threshold adjustment (typically lower than TMAX)
tmean_absolute_threshold = widgets.FloatSlider(value=28.0, min=15.0, max=35.0, step=0.5, description='TMEAN Threshold (°C):')

# Resolution selection for array extraction
resolution_selector = widgets.Dropdown(
    options=[('1km (recommended)', 1000), ('2km', 2000), ('5km', 5000)],
    value=1000,
    description='Resolution:'
)

config_interface = widgets.VBox([
    widgets.HTML('<h3>📊 Analysis Configuration</h3>'),
    widgets.HTML('<div style="background-color: #fff3cd; padding: 10px; border-radius: 5px;">' +
                '<b>TMAX vs TMEAN Comparison:</b><br>' +
                '• <b>TMAX Heat Days:</b> Days when maximum temperature > threshold<br>' +
                '• <b>TMEAN Heat Days:</b> Days when mean temperature > threshold<br>' +
                '• <b>Different thresholds:</b> TMEAN typically uses lower absolute thresholds<br>' +
                '• <b>Same percentile:</b> Both use same percentile of respective distributions</div>'),
    analysis_year,
    widgets.HBox([reference_start, reference_end]),
    widgets.HTML('<b>TMAX Thresholds:</b>'),
    widgets.HBox([absolute_threshold, percentile_threshold]),
    widgets.HTML('<b>TMEAN Thresholds:</b>'),
    tmean_absolute_threshold,
    resolution_selector
])

display(config_interface)
print('📊 Configuration Ready for TMAX vs TMEAN Comparison')

📊 Configuration Ready for TMAX vs TMEAN Comparison


## ⚡ Step 3: Dual Temperature Data Extraction

In [6]:
def get_region_collection(geom):
    """Determine which regional GSHTD collection to use"""
    centroid = geom.centroid().coordinates().getInfo()
    lon, lat = centroid[0], centroid[1]
    
    if lat > 15 and lon > -140 and lon < -40:
        return "projects/sat-io/open-datasets/global-daily-air-temp/north_america"
    elif lat < 35 and lon > -120 and lon < -30:
        return "projects/sat-io/open-datasets/global-daily-air-temp/latin_america"
    elif lat > 30 and lon > -15 and lon < 180:
        return "projects/sat-io/open-datasets/global-daily-air-temp/europe_asia"
    elif lat < 40 and lon > -20 and lon < 55:
        return "projects/sat-io/open-datasets/global-daily-air-temp/africa"
    elif lat < -5 and lon > 110 and lon < 180:
        return "projects/sat-io/open-datasets/global-daily-air-temp/australia"
    else:
        return "projects/sat-io/open-datasets/global-daily-air-temp/north_america"

def get_temperature_collection(region_geom, start_date, end_date, temp_type='tmax'):
    """Get GSHTD temperature collection with minimal preprocessing"""
    collection_id = get_region_collection(region_geom)
    collection = ee.ImageCollection(collection_id)
    
    # Minimal preprocessing - just filter and scale
    filtered_collection = (collection.filterDate(start_date, end_date)
                         .filterBounds(region_geom)
                         .filter(ee.Filter.eq('prop_type', temp_type)))
    
    # Scale to Celsius and clip to ROI
    temp_collection = filtered_collection.map(lambda img: 
        img.select('b1')
          .divide(10)  # Scale to Celsius
          .rename('temperature')
          .clip(region_geom)
          .copyProperties(img, ['system:time_start'])
    )
    
    return temp_collection

def extract_both_temperature_types():
    '''Extract both TMAX and TMEAN temperature arrays from GEE with efficient chunking'''
    global analysis_geom, tmax_data, tmean_data
    
    if analysis_geom is None:
        print('❌ Please set an ROI first!')
        return False
    
    try:
        print('⚡ Starting dual temperature extraction from GEE...')
        
        year = analysis_year.value
        ref_start = reference_start.value
        ref_end = reference_end.value
        scale = resolution_selector.value
        
        area_km2 = analysis_geom.area().divide(1000000).getInfo()
        print(f'   📏 ROI area: {area_km2:.2f} km² at {scale}m resolution')
        
        # Get collections for both temperature types
        print('   📡 Loading GSHTD collections for TMAX and TMEAN...')
        collection_id = get_region_collection(analysis_geom)
        print(f'   📡 Using: {collection_id.split("/")[-1]}')
        
        # Extract TMAX data with chunking
        print('   🔥 Extracting TMAX data...')
        success_tmax = extract_temperature_type_efficient('tmax')
        
        if not success_tmax:
            print('❌ Failed to extract TMAX data')
            return False
        
        # Extract TMEAN data with chunking
        print('   🌡️ Extracting TMEAN data...')
        success_tmean = extract_temperature_type_efficient('tmean')
        
        if not success_tmean:
            print('❌ Failed to extract TMEAN data')
            return False
        
        print(f'\\n✅ Both temperature types extracted successfully!')
        print(f'   🔥 TMAX: {tmax_data.temperature.count().values} observations')
        print(f'   🌡️ TMEAN: {tmean_data.temperature.count().values} observations')
        print(f'   ⚡ Ready for heat risk comparison!')
        
        return True
        
    except Exception as e:
        print(f'❌ Error in dual temperature extraction: {e}')
        return False

def extract_temperature_type_efficient(temp_type):
    '''Extract data for specific temperature type using efficient chunking strategies'''
    global tmax_data, tmean_data
    
    try:
        year = analysis_year.value
        ref_start = reference_start.value
        ref_end = reference_end.value
        scale = resolution_selector.value
        
        # Get collections
        analysis_collection = get_temperature_collection(
            analysis_geom, f'{year}-01-01', f'{year}-12-31', temp_type
        )
        reference_collection = get_temperature_collection(
            analysis_geom, f'{ref_start}-01-01', f'{ref_end}-12-31', temp_type
        )
        
        analysis_count = analysis_collection.size().getInfo()
        reference_count = reference_collection.size().getInfo()
        
        print(f'      📊 {temp_type.upper()} - Analysis: {analysis_count}, Reference: {reference_count}')
        
        if analysis_count == 0 or reference_count == 0:
            print(f'❌ No {temp_type} images found')
            return False
        
        # Estimate data size and choose strategy
        test_image = analysis_collection.first()
        pixel_count = test_image.select('temperature').reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=analysis_geom,
            scale=scale,
            maxPixels=1e9
        ).getInfo()
        
        expected_pixels = pixel_count.get('temperature', 0)
        total_images = analysis_count + reference_count
        estimated_values = expected_pixels * total_images
        
        print(f'      📊 Estimated pixels per image: {expected_pixels:,}')
        print(f'      📊 Total images: {total_images}')
        print(f'      📊 Estimated total values: {estimated_values:,}')
        
        # Choose extraction strategy based on size
        if estimated_values > 1000000:  # Too large for getRegion
            print(f'      🔄 Using temporal chunking strategy for {temp_type}...')
            return extract_with_temporal_chunking_single(temp_type)
        elif estimated_values > 500000:  # Moderate size
            print(f'      🔄 Using annual extraction strategy for {temp_type}...')
            return extract_with_annual_chunks_single(temp_type)
        else:
            print(f'      🔄 Using direct extraction for {temp_type} (small dataset)...')
            return extract_direct_single(temp_type)
        
    except Exception as e:
        print(f'❌ Error in {temp_type} extraction setup: {e}')
        return False

def extract_direct_single(temp_type):
    '''Direct extraction for small datasets'''
    try:
        year = analysis_year.value
        ref_start = reference_start.value
        ref_end = reference_end.value
        scale = resolution_selector.value
        
        # Get collections
        analysis_collection = get_temperature_collection(
            analysis_geom, f'{year}-01-01', f'{year}-12-31', temp_type
        )
        reference_collection = get_temperature_collection(
            analysis_geom, f'{ref_start}-01-01', f'{ref_end}-12-31', temp_type
        )
        
        # Combine for single extraction
        all_collection = analysis_collection.merge(reference_collection)
        
        print(f'      📥 Extracting {temp_type} data in single call...')
        region_data = all_collection.getRegion(
            geometry=analysis_geom,
            scale=scale,
            crs='EPSG:4326'
        ).getInfo()
        
        return process_region_data_single(region_data, temp_type)
        
    except Exception as e:
        print(f'❌ Direct extraction failed for {temp_type}: {e}')
        return False

def extract_with_annual_chunks_single(temp_type):
    '''Extract data year by year'''
    try:
        year = analysis_year.value
        ref_start = reference_start.value
        ref_end = reference_end.value
        scale = resolution_selector.value
        
        all_dataframes = []
        
        # Extract each year separately
        years_to_extract = list(range(ref_start, ref_end + 1)) + [year]
        years_to_extract = sorted(list(set(years_to_extract)))
        
        for extract_year in years_to_extract:
            print(f'      📅 Extracting {temp_type} for year {extract_year}...')
            
            year_collection = get_temperature_collection(
                analysis_geom, f'{extract_year}-01-01', f'{extract_year}-12-31', temp_type
            )
            
            year_count = year_collection.size().getInfo()
            if year_count == 0:
                print(f'         ⚠️ No {temp_type} data for {extract_year}')
                continue
            
            try:
                region_data = year_collection.getRegion(
                    geometry=analysis_geom,
                    scale=scale,
                    crs='EPSG:4326'
                ).getInfo()
                
                if len(region_data) > 1:
                    header = region_data[0]
                    data = region_data[1:]
                    
                    df_year = pd.DataFrame(data, columns=header)
                    df_year['time'] = pd.to_datetime(df_year['time'], unit='ms')
                    df_year = df_year.dropna(subset=['temperature'])
                    df_year['latitude'] = df_year['latitude'].astype(float)
                    df_year['longitude'] = df_year['longitude'].astype(float)
                    df_year['temperature'] = df_year['temperature'].astype(float)
                    
                    all_dataframes.append(df_year)
                    print(f'         ✅ {temp_type} {extract_year}: {len(df_year):,} observations')
                
            except Exception as e:
                print(f'         ❌ Failed to extract {temp_type} {extract_year}: {e}')
                continue
        
        if not all_dataframes:
            print(f'❌ No {temp_type} data extracted')
            return False
        
        # Combine all years
        print(f'      🔗 Combining all {temp_type} years...')
        df = pd.concat(all_dataframes, ignore_index=True)
        
        return finalize_temperature_data_single(df, temp_type)
        
    except Exception as e:
        print(f'❌ Annual chunking failed for {temp_type}: {e}')
        return False

def extract_with_temporal_chunking_single(temp_type):
    '''Extract data with monthly chunks for very large datasets'''
    try:
        year = analysis_year.value
        ref_start = reference_start.value
        ref_end = reference_end.value
        scale = resolution_selector.value
        
        all_dataframes = []
        
        # Create monthly chunks
        years_to_extract = list(range(ref_start, ref_end + 1)) + [year]
        years_to_extract = sorted(list(set(years_to_extract)))
        
        total_months = len(years_to_extract) * 12
        processed_months = 0
        
        for extract_year in years_to_extract:
            for month in range(1, 13):
                print(f'      📅 Extracting {temp_type} {extract_year}-{month:02d} ({processed_months+1}/{total_months})...')
                
                # Monthly date range
                start_date = f'{extract_year}-{month:02d}-01'
                if month == 12:
                    end_date = f'{extract_year+1}-01-01'
                else:
                    end_date = f'{extract_year}-{month+1:02d}-01'
                
                month_collection = get_temperature_collection(
                    analysis_geom, start_date, end_date, temp_type
                )
                
                month_count = month_collection.size().getInfo()
                if month_count == 0:
                    processed_months += 1
                    continue
                
                try:
                    region_data = month_collection.getRegion(
                        geometry=analysis_geom,
                        scale=scale,
                        crs='EPSG:4326'
                    ).getInfo()
                    
                    if len(region_data) > 1:
                        header = region_data[0]
                        data = region_data[1:]
                        
                        df_month = pd.DataFrame(data, columns=header)
                        df_month['time'] = pd.to_datetime(df_month['time'], unit='ms')
                        df_month = df_month.dropna(subset=['temperature'])
                        df_month['latitude'] = df_month['latitude'].astype(float)
                        df_month['longitude'] = df_month['longitude'].astype(float)
                        df_month['temperature'] = df_month['temperature'].astype(float)
                        
                        all_dataframes.append(df_month)
                        print(f'         ✅ {temp_type}: {len(df_month):,} observations')
                    
                except Exception as e:
                    print(f'         ❌ Failed: {e}')
                
                processed_months += 1
                
                # Progress update
                if processed_months % 12 == 0:
                    print(f'      📊 {temp_type}: Completed {processed_months//12} years...')
        
        if not all_dataframes:
            print(f'❌ No {temp_type} data extracted')
            return False
        
        # Combine all chunks
        print(f'      🔗 Combining all {temp_type} temporal chunks...')
        df = pd.concat(all_dataframes, ignore_index=True)
        
        return finalize_temperature_data_single(df, temp_type)
        
    except Exception as e:
        print(f'❌ Temporal chunking failed for {temp_type}: {e}')
        return False

def process_region_data_single(region_data, temp_type):
    '''Process raw region data from getRegion call'''
    try:
        print(f'      🔄 Processing {temp_type} region data: {len(region_data)} rows...')
        
        if len(region_data) <= 1:
            print(f'❌ No {temp_type} data in region extraction')
            return False
        
        header = region_data[0]
        data = region_data[1:]
        
        df = pd.DataFrame(data, columns=header)
        
        return finalize_temperature_data_single(df, temp_type)
        
    except Exception as e:
        print(f'❌ Error processing {temp_type} region data: {e}')
        return False

def finalize_temperature_data_single(df, temp_type):
    '''Convert DataFrame to xarray and finalize for single temperature type'''
    global tmax_data, tmean_data
    
    try:
        # Process data
        df['time'] = pd.to_datetime(df['time'], unit='ms')
        df = df.dropna(subset=['temperature'])
        df['latitude'] = df['latitude'].astype(float)
        df['longitude'] = df['longitude'].astype(float)
        df['temperature'] = df['temperature'].astype(float)
        
        print(f'      📊 {temp_type.upper()} valid observations: {len(df):,}')
        print(f'      📍 {temp_type.upper()} unique pixels: {df[["latitude", "longitude"]].drop_duplicates().shape[0]:,}')
        print(f'      🌡️ {temp_type.upper()} temperature range: {df["temperature"].min():.1f}°C to {df["temperature"].max():.1f}°C')
        
        # Convert to xarray with preserved coordinates
        print(f'      🔄 Converting {temp_type} to xarray dataset...')
        temp_xr = df.set_index(['time', 'latitude', 'longitude']).to_xarray()
        
        # Store in appropriate global variable
        if temp_type == 'tmax':
            tmax_data = temp_xr
        else:
            tmean_data = temp_xr
        
        print(f'      ✅ {temp_type.upper()} array extraction complete!')
        print(f'         📅 Time range: {temp_xr.time.min().values} to {temp_xr.time.max().values}')
        print(f'         🌍 Spatial dims: lat={temp_xr.dims["latitude"]}, lon={temp_xr.dims["longitude"]}')
        print(f'         💾 Memory usage: {temp_xr.nbytes / 1024**2:.1f} MB')
        
        return True
        
    except Exception as e:
        print(f'❌ Error finalizing {temp_type} data: {e}')
        import traceback
        print(f'   Details: {traceback.format_exc()}')
        return False

# Create extraction button
extract_dual_button = widgets.Button(description='📥 Extract TMAX & TMEAN', button_style='primary')
extract_dual_button.on_click(lambda b: extract_both_temperature_types())

dual_extraction_interface = widgets.VBox([
    widgets.HTML('<h3>⚡ Dual Temperature Extraction</h3>'),
    widgets.HTML('<div style="background-color: #e8f4fd; padding: 10px; border-radius: 5px;">' +
                '<b>Two Temperature Types:</b><br>' +
                '• <b>TMAX:</b> Daily maximum temperature (peak heat exposure)<br>' +
                '• <b>TMEAN:</b> Daily mean temperature (sustained heat exposure)<br>' +
                '✨ Both datasets extracted from same GSHTD collection for direct comparison</div>'),
    extract_dual_button
])

display(dual_extraction_interface)
print('⚡ Ready for dual temperature extraction with efficient chunking')

⚡ Ready for dual temperature extraction with efficient chunking


## 🔥 Step 4: Heat Risk Calculations (TMAX & TMEAN)

In [ ]:
def calculate_heat_risk_both_methods():
    '''Calculate heat risk for both TMAX and TMEAN approaches'''
    global tmax_data, tmean_data, heat_risk_results
    
    if tmax_data is None or tmean_data is None:
        print('❌ Please extract temperature data first!')
        return False
    
    try:
        print('🔥 Calculating heat risk for both TMAX and TMEAN...')
        
        year = analysis_year.value
        ref_start = reference_start.value
        ref_end = reference_end.value
        tmax_abs_threshold = absolute_threshold.value
        tmean_abs_threshold = tmean_absolute_threshold.value
        pct_threshold = percentile_threshold.value
        
        print(f'   🔥 TMAX absolute threshold: {tmax_abs_threshold}°C')
        print(f'   🌡️ TMEAN absolute threshold: {tmean_abs_threshold}°C')
        print(f'   📊 Percentile threshold: {pct_threshold}%')
        
        # Calculate TMAX heat risk
        print('   🔥 Processing TMAX heat risk...')
        tmax_results = calculate_heat_risk_single(tmax_data, 'TMAX', tmax_abs_threshold, pct_threshold, year, ref_start, ref_end)
        
        # Calculate TMEAN heat risk
        print('   🌡️ Processing TMEAN heat risk...')
        tmean_results = calculate_heat_risk_single(tmean_data, 'TMEAN', tmean_abs_threshold, pct_threshold, year, ref_start, ref_end)
        
        if not tmax_results or not tmean_results:
            print('❌ Failed to calculate heat risk for one or both methods')
            return False
        
        # Store results with method prefixes
        heat_risk_results = {}
        for key, value in tmax_results.items():
            heat_risk_results[f'tmax_{key}'] = value
        for key, value in tmean_results.items():
            heat_risk_results[f'tmean_{key}'] = value
        
        # Calculate comparison metrics
        print('   📊 Calculating comparison metrics...')
        heat_risk_results['heat_days_difference'] = tmax_results['heat_days'] - tmean_results['heat_days']
        heat_risk_results['heat_days_ratio'] = xr.where(
            tmean_results['heat_days'] > 0,
            tmax_results['heat_days'] / tmean_results['heat_days'],
            np.nan
        )
        
        print(f'\n✅ Heat risk calculations complete!')
        
        # Print comparison summary
        tmax_mean = tmax_results['heat_days'].mean().values
        tmean_mean = tmean_results['heat_days'].mean().values
        
        print(f'\n📊 HEAT RISK COMPARISON:')
        print(f'   🔥 TMAX Heat Days - Mean: {tmax_mean:.1f}, Max: {tmax_results["heat_days"].max().values:.0f}')
        print(f'   🌡️ TMEAN Heat Days - Mean: {tmean_mean:.1f}, Max: {tmean_results["heat_days"].max().values:.0f}')
        print(f'   📈 Difference - Mean: {(tmax_mean - tmean_mean):.1f} days')
        
        # Count pixels with heat days
        tmax_pixels = (tmax_results['heat_days'] > 0).sum().values
        tmean_pixels = (tmean_results['heat_days'] > 0).sum().values
        total_pixels = tmax_results['heat_days'].count().values
        
        print(f'\n🎯 AFFECTED PIXELS:')
        print(f'   🔥 TMAX: {tmax_pixels} of {total_pixels} pixels ({tmax_pixels/total_pixels*100:.1f}%)')
        print(f'   🌡️ TMEAN: {tmean_pixels} of {total_pixels} pixels ({tmean_pixels/total_pixels*100:.1f}%)')
        
        return True
        
    except Exception as e:
        print(f'❌ Error in heat risk calculation: {e}')
        import traceback
        print(f'   Details: {traceback.format_exc()}')
        return False

def calculate_heat_risk_single(temp_data, method_name, abs_threshold, pct_threshold, year, ref_start, ref_end):
    '''Calculate heat risk for a single temperature dataset'''
    try:
        # Filter data efficiently  
        analysis_data = temp_data.sel(time=str(year))
        reference_data = temp_data.sel(time=slice(f'{ref_start}-01-01', f'{ref_end}-12-31'))
        
        print(f'      📅 {method_name} - Analysis: {len(analysis_data.time)} days, Reference: {len(reference_data.time)} days')
        
        # Calculate reference percentile
        print(f'      ⚡ Calculating {method_name} reference percentile...')
        reference_percentile = reference_data.temperature.quantile(pct_threshold/100, dim='time')
        
        # Determine threshold (max of percentile and absolute)
        print(f'      ⚡ Calculating {method_name} heat days...')
        threshold = xr.where(reference_percentile > abs_threshold, reference_percentile, abs_threshold)
        heat_days = (analysis_data.temperature > threshold).sum(dim='time').fillna(0)
        
        # Additional statistics
        print(f'      ⚡ Calculating {method_name} statistics...')
        annual_max = analysis_data.temperature.max(dim='time').fillna(0)
        annual_min = analysis_data.temperature.min(dim='time').fillna(0)
        annual_mean = analysis_data.temperature.mean(dim='time').fillna(0)
        
        return {
            'heat_days': heat_days,
            'reference_percentile': reference_percentile,
            'threshold_used': threshold,
            'annual_max': annual_max,
            'annual_min': annual_min,
            'annual_mean': annual_mean
        }
        
    except Exception as e:
        print(f'❌ Error calculating {method_name} heat risk: {e}')
        return None

# Create processing button
process_heat_risk_button = widgets.Button(description='🔥 Calculate Heat Risk', button_style='success')
process_heat_risk_button.on_click(lambda b: calculate_heat_risk_both_methods())

heat_risk_interface = widgets.VBox([
    widgets.HTML('<h3>🔥 Heat Risk Calculations</h3>'),
    widgets.HTML('<div style="background-color: #d4edda; padding: 10px; border-radius: 5px;">' +
                '<b>Dual Heat Risk Calculation:</b><br>' +
                '• Both methods use same percentile approach<br>' +
                '• Different absolute thresholds reflect different temperature types<br>' +
                '• Results directly comparable for analysis<br>' +
                '• Comparison metrics automatically calculated</div>'),
    process_heat_risk_button
])

display(heat_risk_interface)
print('🔥 Ready for dual heat risk calculation')

## 🗺️ Step 5: Side-by-Side Comparison Visualization

In [ ]:
def create_comparison_visualization():
    '''Create comprehensive side-by-side comparison of TMAX vs TMEAN heat risk'''
    if not heat_risk_results:
        print('❌ No heat risk results available. Please calculate heat risk first!')
        return
    
    print('🗺️ Creating TMAX vs TMEAN comparison visualization...')
    
    # Main comparison: Heat days side by side
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Row 1: Heat Days Comparison
    if 'tmax_heat_days' in heat_risk_results and 'tmean_heat_days' in heat_risk_results:
        # TMAX Heat Days
        im1 = heat_risk_results['tmax_heat_days'].plot(
            ax=axes[0,0], cmap='Reds', add_colorbar=True
        )
        axes[0,0].set_title('🔥 TMAX Heat Days', fontweight='bold', fontsize=14)
        axes[0,0].set_xlabel('Longitude')
        axes[0,0].set_ylabel('Latitude')
        
        # TMEAN Heat Days
        im2 = heat_risk_results['tmean_heat_days'].plot(
            ax=axes[0,1], cmap='Reds', add_colorbar=True
        )
        axes[0,1].set_title('🌡️ TMEAN Heat Days', fontweight='bold', fontsize=14)
        axes[0,1].set_xlabel('Longitude')
        axes[0,1].set_ylabel('Latitude')
        
        # Difference (TMAX - TMEAN)
        if 'heat_days_difference' in heat_risk_results:
            im3 = heat_risk_results['heat_days_difference'].plot(
                ax=axes[0,2], cmap='RdBu_r', add_colorbar=True, center=0
            )
            axes[0,2].set_title('📊 Difference (TMAX - TMEAN)', fontweight='bold', fontsize=14)
            axes[0,2].set_xlabel('Longitude')
            axes[0,2].set_ylabel('Latitude')
    
    # Row 2: Annual Mean Temperatures and Thresholds
    if 'tmax_annual_mean' in heat_risk_results and 'tmean_annual_mean' in heat_risk_results:
        # TMAX Annual Mean
        im4 = heat_risk_results['tmax_annual_mean'].plot(
            ax=axes[1,0], cmap='RdYlBu_r', add_colorbar=True
        )
        axes[1,0].set_title('🔥 TMAX Annual Mean Temperature', fontweight='bold', fontsize=14)
        axes[1,0].set_xlabel('Longitude')
        axes[1,0].set_ylabel('Latitude')
        
        # TMEAN Annual Mean
        im5 = heat_risk_results['tmean_annual_mean'].plot(
            ax=axes[1,1], cmap='RdYlBu_r', add_colorbar=True
        )
        axes[1,1].set_title('🌡️ TMEAN Annual Mean Temperature', fontweight='bold', fontsize=14)
        axes[1,1].set_xlabel('Longitude')
        axes[1,1].set_ylabel('Latitude')
        
        # Temperature Range (TMAX - TMEAN annual means)
        temp_range = heat_risk_results['tmax_annual_mean'] - heat_risk_results['tmean_annual_mean']
        im6 = temp_range.plot(
            ax=axes[1,2], cmap='plasma', add_colorbar=True
        )
        axes[1,2].set_title('🌡️ Annual Temp Range (TMAX - TMEAN)', fontweight='bold', fontsize=14)
        axes[1,2].set_xlabel('Longitude')
        axes[1,2].set_ylabel('Latitude')
    
    plt.tight_layout()
    plt.show()
    
    # Create statistical comparison
    create_statistical_comparison()
    
    print('✅ Comparison visualization complete!')

def create_statistical_comparison():
    '''Create statistical comparison between TMAX and TMEAN results'''
    
    print('\n📊 STATISTICAL COMPARISON:')
    print('='*60)
    
    # Heat days comparison
    if 'tmax_heat_days' in heat_risk_results and 'tmean_heat_days' in heat_risk_results:
        tmax_hd = heat_risk_results['tmax_heat_days']
        tmean_hd = heat_risk_results['tmean_heat_days']
        
        print('🔥 HEAT DAYS STATISTICS:')
        print(f'   TMAX  - Mean: {tmax_hd.mean().values:.1f}, Std: {tmax_hd.std().values:.1f}, Max: {tmax_hd.max().values:.0f}')
        print(f'   TMEAN - Mean: {tmean_hd.mean().values:.1f}, Std: {tmean_hd.std().values:.1f}, Max: {tmean_hd.max().values:.0f}')
        
        # Pixels with heat days
        tmax_affected = (tmax_hd > 0).sum().values
        tmean_affected = (tmean_hd > 0).sum().values
        total_pixels = tmax_hd.count().values
        
        print(f'\n🎯 AFFECTED AREAS:')
        print(f'   TMAX:  {tmax_affected:,} pixels ({tmax_affected/total_pixels*100:.1f}%)')
        print(f'   TMEAN: {tmean_affected:,} pixels ({tmean_affected/total_pixels*100:.1f}%)')
        
        # Correlation
        correlation = xr.corr(tmax_hd, tmean_hd).values
        print(f'\n🔗 CORRELATION: {correlation:.3f}')
    
    # Temperature comparison
    if 'tmax_annual_mean' in heat_risk_results and 'tmean_annual_mean' in heat_risk_results:
        tmax_temp = heat_risk_results['tmax_annual_mean']
        tmean_temp = heat_risk_results['tmean_annual_mean']
        
        print('\n🌡️ TEMPERATURE STATISTICS:')
        print(f'   TMAX  Annual Mean - Mean: {tmax_temp.mean().values:.1f}°C, Range: {tmax_temp.min().values:.1f} to {tmax_temp.max().values:.1f}°C')
        print(f'   TMEAN Annual Mean - Mean: {tmean_temp.mean().values:.1f}°C, Range: {tmean_temp.min().values:.1f} to {tmean_temp.max().values:.1f}°C')
        
        temp_diff = tmax_temp - tmean_temp
        print(f'   Temperature Difference - Mean: {temp_diff.mean().values:.1f}°C, Std: {temp_diff.std().values:.1f}°C')
    
    # Threshold comparison
    if 'tmax_threshold_used' in heat_risk_results and 'tmean_threshold_used' in heat_risk_results:
        tmax_thresh = heat_risk_results['tmax_threshold_used']
        tmean_thresh = heat_risk_results['tmean_threshold_used']
        
        print('\n🎯 THRESHOLD COMPARISON:')
        print(f'   TMAX  Threshold - Mean: {tmax_thresh.mean().values:.1f}°C')
        print(f'   TMEAN Threshold - Mean: {tmean_thresh.mean().values:.1f}°C')
        print(f'   Threshold Difference: {(tmax_thresh.mean() - tmean_thresh.mean()).values:.1f}°C')

def create_scatter_analysis():
    '''Create scatter plot analysis of TMAX vs TMEAN heat days'''
    if 'tmax_heat_days' not in heat_risk_results or 'tmean_heat_days' not in heat_risk_results:
        print('❌ Heat days data not available for scatter analysis')
        return
    
    print('📊 Creating scatter plot analysis...')
    
    # Extract data for plotting
    tmax_hd = heat_risk_results['tmax_heat_days'].values.flatten()
    tmean_hd = heat_risk_results['tmean_heat_days'].values.flatten()
    
    # Remove NaN values
    valid_mask = ~(np.isnan(tmax_hd) | np.isnan(tmean_hd))
    tmax_valid = tmax_hd[valid_mask]
    tmean_valid = tmean_hd[valid_mask]
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Scatter plot
    axes[0].scatter(tmean_valid, tmax_valid, alpha=0.6, s=1)
    axes[0].plot([0, max(tmean_valid.max(), tmax_valid.max())], 
                [0, max(tmean_valid.max(), tmax_valid.max())], 
                'r--', alpha=0.8, label='1:1 Line')
    axes[0].set_xlabel('TMEAN Heat Days')
    axes[0].set_ylabel('TMAX Heat Days')
    axes[0].set_title('🔥 TMAX vs TMEAN Heat Days')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Histogram of differences
    difference = tmax_valid - tmean_valid
    axes[1].hist(difference, bins=50, alpha=0.7, color='coral', edgecolor='black')
    axes[1].axvline(difference.mean(), color='red', linestyle='--', 
                   label=f'Mean: {difference.mean():.1f} days')
    axes[1].axvline(0, color='black', linestyle='-', alpha=0.5)
    axes[1].set_xlabel('Difference (TMAX - TMEAN Heat Days)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('📊 Distribution of Heat Days Difference')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    correlation = np.corrcoef(tmax_valid, tmean_valid)[0, 1]
    print(f'\n📊 SCATTER ANALYSIS:')
    print(f'   Correlation coefficient: {correlation:.3f}')
    print(f'   Mean difference: {difference.mean():.1f} ± {difference.std():.1f} days')
    print(f'   Pixels where TMAX > TMEAN: {(difference > 0).sum()} ({(difference > 0).mean()*100:.1f}%)')
    print(f'   Pixels where TMEAN > TMAX: {(difference < 0).sum()} ({(difference < 0).mean()*100:.1f}%)')

# Create visualization buttons
comparison_viz_button = widgets.Button(description='🗺️ Create Comparison Maps', button_style='info')
scatter_analysis_button = widgets.Button(description='📊 Scatter Analysis', button_style='warning')

comparison_viz_button.on_click(lambda b: create_comparison_visualization())
scatter_analysis_button.on_click(lambda b: create_scatter_analysis())

visualization_interface = widgets.VBox([
    widgets.HTML('<h3>🗺️ TMAX vs TMEAN Comparison Visualization</h3>'),
    widgets.HTML('<div style="background-color: #fff3cd; padding: 10px; border-radius: 5px;">' +
                '<b>Comprehensive Comparison:</b><br>' +
                '• <b>Side-by-side maps:</b> Visual comparison of heat risk patterns<br>' +
                '• <b>Difference analysis:</b> Where methods disagree most<br>' +
                '• <b>Statistical comparison:</b> Quantitative analysis<br>' +
                '• <b>Scatter analysis:</b> Pixel-by-pixel correlation</div>'),
    widgets.HBox([comparison_viz_button, scatter_analysis_button])
])

display(visualization_interface)
print('🗺️ Ready for comprehensive TMAX vs TMEAN comparison')

## 📁 Step 6: Export Comparison Results

In [ ]:
def export_comparison_results():
    '''Export all comparison results to files'''
    if not heat_risk_results:
        print('❌ No heat risk results available. Please calculate heat risk first!')
        return
    
    try:
        print('📁 Exporting TMAX vs TMEAN comparison results...')
        
        year = analysis_year.value
        
        # Create comparison summary statistics
        summary_data = []
        
        for metric_name, dataset in heat_risk_results.items():
            if hasattr(dataset, 'mean') and len(dataset.dims) <= 2:
                try:
                    valid_data = dataset.values[~np.isnan(dataset.values)]
                    
                    if len(valid_data) > 0:
                        summary_data.append({
                            'metric': metric_name,
                            'method': 'TMAX' if 'tmax' in metric_name else 'TMEAN' if 'tmean' in metric_name else 'COMPARISON',
                            'valid_pixels': len(valid_data),
                            'mean': float(np.mean(valid_data)),
                            'min': float(np.min(valid_data)),
                            'max': float(np.max(valid_data)),
                            'std': float(np.std(valid_data)),
                            'median': float(np.median(valid_data))
                        })
                except Exception as e:
                    print(f'     ⚠️ Skipping {metric_name}: {e}')
        
        # Save summary table
        if summary_data:
            summary_df = pd.DataFrame(summary_data)
            summary_file = f'../outputs/tmax_tmean_comparison_summary_{year}.csv'
            summary_df.to_csv(summary_file, index=False)
            print(f'   ✅ Comparison summary: {summary_file}')
            
            # Display summary
            print('\n📊 COMPARISON SUMMARY:')
            display(summary_df.groupby('method').agg({
                'mean': 'mean',
                'min': 'min', 
                'max': 'max',
                'std': 'mean'
            }).round(2))
        
        # Export as NetCDF with proper CRS
        print('\n   📦 Creating comparison NetCDF...')
        
        # Create xarray Dataset from results
        spatial_results = {}
        for k, v in heat_risk_results.items():
            if hasattr(v, 'dims') and 'latitude' in v.dims and 'longitude' in v.dims:
                if len(v.dims) == 2:  # Spatial data only
                    spatial_results[k] = v.fillna(0)
        
        if spatial_results:
            results_ds = xr.Dataset(spatial_results)
            
            # Add proper CRS information
            results_ds.latitude.attrs['standard_name'] = 'latitude'
            results_ds.latitude.attrs['long_name'] = 'latitude'
            results_ds.latitude.attrs['units'] = 'degrees_north'
            results_ds.latitude.attrs['axis'] = 'Y'
            
            results_ds.longitude.attrs['standard_name'] = 'longitude'
            results_ds.longitude.attrs['long_name'] = 'longitude'
            results_ds.longitude.attrs['units'] = 'degrees_east'
            results_ds.longitude.attrs['axis'] = 'X'
            
            # Add CRS variable
            crs = xr.DataArray(
                data=np.int32(1),
                attrs={
                    'grid_mapping_name': 'latitude_longitude',
                    'longitude_of_prime_meridian': 0.0,
                    'semi_major_axis': 6378137.0,
                    'inverse_flattening': 298.257223563
                }
            )
            results_ds['crs'] = crs
            
            # Add grid_mapping to all data variables
            for var_name in results_ds.data_vars:
                if var_name != 'crs':
                    results_ds[var_name].attrs['grid_mapping'] = 'crs'
            
            # Add metadata
            results_ds.attrs['analysis_year'] = year
            results_ds.attrs['reference_period'] = f'{reference_start.value}-{reference_end.value}'
            results_ds.attrs['created'] = datetime.now().isoformat()
            results_ds.attrs['method'] = 'tmax_tmean_comparison'
            results_ds.attrs['tmax_absolute_threshold'] = absolute_threshold.value
            results_ds.attrs['tmean_absolute_threshold'] = tmean_absolute_threshold.value
            results_ds.attrs['percentile_threshold'] = percentile_threshold.value
            results_ds.attrs['resolution'] = f'{resolution_selector.value}m'
            results_ds.attrs['crs'] = 'EPSG:4326'
            
            netcdf_file = f'../outputs/tmax_tmean_comparison_{year}.nc'
            results_ds.to_netcdf(netcdf_file)
            
            print(f'   ✅ Comparison NetCDF: {netcdf_file}')
            print(f'      Variables: {len(results_ds.data_vars)} ({list(results_ds.data_vars)[:5]}...)')
            print(f'      Dimensions: {dict(results_ds.dims)}')
            print(f'      File size: {os.path.getsize(netcdf_file) / 1024**2:.1f} MB')
        
        # Export key comparison GeoTIFF files
        print('\n   🗺️ Exporting key comparison GeoTIFFs...')
        
        key_outputs = {
            'tmax_heat_days': 'TMAX Heat Days',
            'tmean_heat_days': 'TMEAN Heat Days', 
            'heat_days_difference': 'Heat Days Difference (TMAX-TMEAN)',
            'heat_days_ratio': 'Heat Days Ratio (TMAX/TMEAN)'
        }
        
        for metric_key, description in key_outputs.items():
            if metric_key in heat_risk_results:
                try:
                    dataset = heat_risk_results[metric_key].fillna(0)
                    dataset.rio.write_crs("EPSG:4326", inplace=True)
                    
                    output_file = f'../outputs/{metric_key}_{year}.tif'
                    dataset.rio.to_raster(output_file)
                    
                    print(f'      ✅ {description} → {os.path.basename(output_file)}')
                    
                except Exception as e:
                    print(f'      ⚠️ Failed to export {metric_key}: {e}')
        
        print(f'\n✅ Export complete! Files saved to ../outputs/')
        print(f'   📊 Summary: tmax_tmean_comparison_summary_{year}.csv')
        print(f'   📦 NetCDF: tmax_tmean_comparison_{year}.nc')
        print(f'   🗺️ Key GeoTIFFs: *_heat_days_{year}.tif, heat_days_difference_{year}.tif')
        print(f'   ✨ All files have proper WGS84 projection for GIS use!')
        
    except Exception as e:
        print(f'❌ Error exporting comparison results: {e}')
        import traceback
        print(f'   Details: {traceback.format_exc()}')

# Create export button
export_comparison_button = widgets.Button(description='📁 Export Comparison Results', button_style='warning')
export_comparison_button.on_click(lambda b: export_comparison_results())

export_interface = widgets.VBox([
    widgets.HTML('<h3>📁 Export Comparison Results</h3>'),
    widgets.HTML('<div style="background-color: #d4edda; padding: 10px; border-radius: 5px;">' +
                '<b>Complete Export Package:</b><br>' +
                '• Statistical summary comparing both methods<br>' +
                '• NetCDF with all calculated metrics<br>' +
                '• Key GeoTIFF files for GIS analysis<br>' +
                '• Proper spatial reference information</div>'),
    export_comparison_button
])

display(export_interface)
print('📁 Ready to export TMAX vs TMEAN comparison results')

## 🎯 Summary: TMAX vs TMEAN Heat Risk Comparison

This notebook provides a **comprehensive comparison** between TMAX and TMEAN-based heat risk calculations:

### 🔬 **Research Questions Addressed:**

**1. Heat Risk Pattern Differences**
- How do spatial patterns of heat risk differ between methods?
- Which areas are identified as high-risk by each approach?
- Where do the methods disagree most?

**2. Sensitivity Analysis**  
- How sensitive are results to threshold choices?
- Which method identifies more heat-vulnerable areas?
- What's the correlation between TMAX and TMEAN heat days?

### 💡 **Key Insights:**

**TMAX Heat Risk (🔥)**
- Captures **peak daily heat exposure**
- Better for identifying extreme heat events
- More sensitive to afternoon/evening heat spikes
- Traditional approach in heat health studies

**TMEAN Heat Risk (🌡️)**
- Captures **sustained heat throughout day**
- Better for identifying persistent heat exposure
- Includes nighttime temperature effects
- May better reflect physiological heat stress

### 🚀 **Methodological Advantages:**
- **Same data source:** Both use GSHTD for consistency
- **Same percentile approach:** Comparable threshold methodology
- **Direct comparison:** Pixel-by-pixel analysis
- **Multiple outputs:** Maps, statistics, and correlation analysis

### 📊 **Applications:**
- **Urban planning:** Identify vulnerable neighborhoods
- **Public health:** Compare heat exposure metrics
- **Climate adaptation:** Understand different heat risk profiles
- **Research:** Validate heat risk methodologies

This comparison helps determine which temperature metric better captures heat vulnerability in your specific study area! 🎯